**Item Recommendation**

**Product Recommendations on Product Pages:**

Show "Similar Products" when a user views a product.
Search Enhancements:

Improve search results by suggesting alternatives when a product isn't found.

**Provide Detailed Recommendations**

Each recommendation includes:

Product Name: The name of the recommended product.

Similarity Score: A measure of how similar the recommended product is to the input product.

Average Price: The average price of the product across stores.

Store Availability: A list of stores where the product is available.

**Input**

Accepts a product name (e.g., "Chocolate") as the primary input.

Optionally accepts a top_n parameter to define how many recommendations to return.

**Output**

A list of recommended products with associated details.

**Handles Edge Cases:**

If the product name doesn't match exactly, it suggests partial matches.

Example: If "Choco" is entered, it might suggest "Chocolate Bar" or "Choco Chips".

In [19]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import json

In [2]:
# Load the cleaned datasets
aldi_data = pd.read_csv('Aldi_Cleaned.csv')
coles_data = pd.read_csv('Coles_Cleaned.csv')
iga_data = pd.read_csv('IGA_Cleaned.csv')
woolworths_data = pd.read_csv('Woolworths_Cleaned.csv')

In [3]:
# Standardizing and renaming columns for consistency
def standardize_columns(df, store_name):
    df = df.rename(columns={
        'Product Name': 'Product',
        'Brand_Product_Size': 'Product',
        'Price': 'Price',
        'Unit Price': 'ppu',
        'COL Price': 'Price',
        'COL ppu': 'ppu',
        'IGA Price': 'Price',
        'IGA ppu': 'ppu',
        'WOW Price': 'Price',
        'WOW ppu': 'ppu',
        'Aldi Category': 'Category',
        'COL Category': 'Category',
        'IGA Category': 'Category',
        'WOW Category': 'Category',
        'URL': 'Product_URL',
        'Product URL': 'Product_URL'
    })
    df = df[['Product', 'Price', 'ppu', 'Category', 'Product_URL']]
    df.columns = [f'{col}_{store_name}' if col != 'Product' else col for col in df.columns]
    return df

In [4]:
# Standardizing each dataset
aldi_standardized = standardize_columns(aldi_data, 'Aldi')
coles_standardized = standardize_columns(coles_data, 'Coles')
iga_standardized = standardize_columns(iga_data, 'IGA')
woolworths_standardized = standardize_columns(woolworths_data, 'Woolworths')

In [5]:
# Resolving duplicate columns in Coles and Woolworths datasets
coles_standardized = coles_standardized.loc[:, ~coles_standardized.columns.duplicated()]
woolworths_standardized = woolworths_standardized.loc[:, ~woolworths_standardized.columns.duplicated()]

In [6]:
merged_data = pd.concat([
    aldi_standardized,
    coles_standardized,
    iga_standardized,
    woolworths_standardized
], ignore_index=True)

In [7]:
# Display the first few rows of the merged data
merged_data.head()

,Product,Price_Aldi,ppu_Aldi,Category_Aldi,Product_URL_Aldi,Price_Coles,ppu_Coles,Category_Coles,Product_URL_Coles,Price_IGA,ppu_IGA,Category_IGA,Product_URL_IGA,Price_Woolworths,ppu_Woolworths,Category_Woolworths,Product_URL_Woolworths
0,Essential Health Paw Paw Ointment ea,2.79,$11.16 per 100g,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Hedanol Paracetamol Capsule Shaped Tablets 20pk,69.0,3c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Essential Health Effervescents 15pk ea,4.49,30c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Essential Health Electrolytes 20pk 20pk,5.49,27c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Hedafen Ibuprofen Liquid Capsules 20pk 20pk,2.39,12c per capsule,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Adding the 'Store Availability' column
def get_store_availability(row):
    available_stores = []
    if not pd.isna(row['Price_Aldi']):
        available_stores.append('Aldi')
    if not pd.isna(row['Price_Coles']):
        available_stores.append('Coles')
    if not pd.isna(row['Price_IGA']):
        available_stores.append('IGA')
    if not pd.isna(row['Price_Woolworths']):
        available_stores.append('Woolworths')
    return ', '.join(available_stores) if available_stores else 'Not Available'

merged_data['Store_Availability'] = merged_data.apply(get_store_availability, axis=1)

In [9]:
merged_data.head()

,Product,Price_Aldi,ppu_Aldi,Category_Aldi,Product_URL_Aldi,Price_Coles,ppu_Coles,Category_Coles,Product_URL_Coles,Price_IGA,ppu_IGA,Category_IGA,Product_URL_IGA,Price_Woolworths,ppu_Woolworths,Category_Woolworths,Product_URL_Woolworths,Store_Availability
0,Essential Health Paw Paw Ointment ea,2.79,$11.16 per 100g,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Aldi
1,Hedanol Paracetamol Capsule Shaped Tablets 20pk,69.0,3c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Aldi
2,Essential Health Effervescents 15pk ea,4.49,30c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Aldi
3,Essential Health Electrolytes 20pk 20pk,5.49,27c per tablet,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Aldi
4,Hedafen Ibuprofen Liquid Capsules 20pk 20pk,2.39,12c per capsule,Health & Beauty,https://www.aldi.com.au/en/groceries/health/he...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Aldi


Normalize the price and handle missing values

In [10]:
# Step 1: Normalize Prices
def normalize_price(price):
    try:
        # Extract numeric value from price, handle cases like '$1.23 per 100g'
        numeric_price = float(''.join(filter(lambda x: x.isdigit() or x == '.', str(price))))
        return numeric_price
    except:
        return np.nan

# Apply normalization to all price columns
merged_data['Price_Aldi'] = merged_data['Price_Aldi'].apply(normalize_price)
merged_data['Price_Coles'] = merged_data['Price_Coles'].apply(normalize_price)
merged_data['Price_IGA'] = merged_data['Price_IGA'].apply(normalize_price)
merged_data['Price_Woolworths'] = merged_data['Price_Woolworths'].apply(normalize_price)

In [11]:
# Step 2: Handle Missing Values
def impute_missing_prices(row):
    # Fill NaN prices with the average of available prices across stores
    prices = [row['Price_Aldi'], row['Price_Coles'], row['Price_IGA'], row['Price_Woolworths']]
    available_prices = [p for p in prices if not pd.isna(p)]
    if available_prices:
        return np.mean(available_prices)
    return np.nan

merged_data['Average_Price'] = merged_data.apply(impute_missing_prices, axis=1)

In [12]:
print(merged_data.head())

                                            Product  Price_Aldi  \
0              Essential Health Paw Paw Ointment ea        2.79   
1  Hedanol Paracetamol Capsule Shaped Tablets 20pk        69.00   
2            Essential Health Effervescents 15pk ea        4.49   
3           Essential Health Electrolytes 20pk 20pk        5.49   
4       Hedafen Ibuprofen Liquid Capsules 20pk 20pk        2.39   

          ppu_Aldi    Category_Aldi  \
0  $11.16 per 100g  Health & Beauty   
1    3c per tablet  Health & Beauty   
2   30c per tablet  Health & Beauty   
3   27c per tablet  Health & Beauty   
4  12c per capsule  Health & Beauty   

                                    Product_URL_Aldi  Price_Coles ppu_Coles  \
0  https://www.aldi.com.au/en/groceries/health/he...          NaN       NaN   
1  https://www.aldi.com.au/en/groceries/health/he...          NaN       NaN   
2  https://www.aldi.com.au/en/groceries/health/he...          NaN       NaN   
3  https://www.aldi.com.au/en/groceries/health

Feature Matrix Construction

In [14]:
# Step 3: Feature Matrix Construction
# One-hot encode categories
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
category_column = merged_data['Category_Aldi'].fillna('Unknown').values.reshape(-1, 1)
category_encoded = encoder.fit_transform(category_column)

# Normalize numerical features (Average Price)
scaler = StandardScaler()
numerical_features = scaler.fit_transform(merged_data[['Average_Price']].fillna(0))

# Include Store Availability as an additional feature
store_availability_encoded = pd.get_dummies(merged_data['Store_Availability']).values

# Combine all features into a feature matrix
feature_matrix = np.hstack((category_encoded, numerical_features, store_availability_encoded))

# Pre-compute weighted feature matrix for efficiency
weights = np.array([1.0] * category_encoded.shape[1] + [1.5] * numerical_features.shape[1] + [1.2] * store_availability_encoded.shape[1])
adjusted_feature_matrix = feature_matrix * weights
adjusted_similarity_matrix = cosine_similarity(adjusted_feature_matrix)

c:\Users\NISHANT KHAMKAR\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Model development

In [16]:
# Step 4: Content-Based Recommendation
# Function to recommend products
def recommend_products_with_weights(product_index, top_n=5):
    similarity_scores = list(enumerate(adjusted_similarity_matrix[product_index]))
    # Sort by similarity score
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    # Get the top_n similar products
    top_products = similarity_scores[1:top_n+1]
    # Return product details with similarity scores
    recommendations = []
    for i, score in top_products:
        product = merged_data.iloc[i]
        recommendations.append({
            'Product': product['Product'],
            'Similarity Score': score,
            'Average Price': product['Average_Price'],
            'Store Availability': product['Store_Availability']
        })
    return recommendations

user input for recommendation

In [17]:
# Step 5: User Input for Recommendations
def get_product_index(product_name):
    try:
        return merged_data[merged_data['Product'].str.contains(product_name, case=False, na=False)].index[0]
    except IndexError:
        return None

In [18]:
# Example: User provides a product name
product_name = "Chocolate"
product_index = get_product_index(product_name)
if product_index is not None:
    recommended_products = recommend_products_with_weights(product_index, top_n=5)
    print(f"Recommended Products for '{product_name}':")
    for product in recommended_products:
        print(product)
else:
    print(f"Product '{product_name}' not found. Did you mean:")
    suggestions = merged_data[merged_data['Product'].str.contains(product_name[:3], case=False, na=False)].head()
    print(suggestions['Product'].tolist())

Recommended Products for 'Chocolate':
{'Product': 'Dentitex Orbit Toothbrush 2pk 2pk', 'Similarity Score': 1.0, 'Average Price': 2.99, 'Store Availability': 'Aldi'}
{'Product': 'Essential Health Paw Paw Ointment ea', 'Similarity Score': 0.9998546196751059, 'Average Price': 2.79, 'Store Availability': 'Aldi'}
{'Product': 'Dentitex Mouthwash 500ml 500ml', 'Similarity Score': 0.9990405090754609, 'Average Price': 3.49, 'Store Availability': 'Aldi'}
{'Product': 'Hedafen Ibuprofen Liquid Capsules 20pk 20pk', 'Similarity Score': 0.9987338260619745, 'Average Price': 2.39, 'Store Availability': 'Aldi'}
{'Product': 'LACURA® Essentials Fine Balance Foaming Cleansing Gel 150ml ', 'Similarity Score': 0.9974890200328712, 'Average Price': 3.79, 'Store Availability': 'Aldi'}


The model is capable of providing:

Accurate product recommendations based on product features.

Detailed outputs with price and availability information.

Flexibility to handle incomplete data and suggest alternatives.

Scalability for integration into your Node.js backend and front-end web application.

In [20]:
# Save the similarity matrix
np.save('adjusted_similarity_matrix.npy', adjusted_similarity_matrix)

# Save the product metadata (e.g., Product names and details)
product_metadata = merged_data[['Product', 'Average_Price', 'Store_Availability']].to_dict(orient='records')
with open('product_metadata.json', 'w') as f:
    json.dump(product_metadata, f)

Saving the metadata similarity matrix

In [2]:
import numpy as np
import json

# Load the matrix
similarity_matrix = np.load("adjusted_similarity_matrix.npy")

# Define the chunk size
chunk_size = 1000  # Adjust based on available memory
num_rows = similarity_matrix.shape[0]

# Save each chunk as a separate JSON file
for i in range(0, num_rows, chunk_size):
    chunk = similarity_matrix[i:i+chunk_size].tolist()
    with open(f"similarity_matrix_chunk_{i//chunk_size}.json", "w") as f:
        json.dump(chunk, f)
    print(f"Chunk {i//chunk_size} saved.")

print("Matrix split and saved as JSON chunks.")

Chunk 0 saved.
Chunk 1 saved.
Chunk 2 saved.
Chunk 3 saved.
Chunk 4 saved.
Chunk 5 saved.
Chunk 6 saved.
Chunk 7 saved.
Chunk 8 saved.
Chunk 9 saved.
Chunk 10 saved.
Chunk 11 saved.
Chunk 12 saved.
Chunk 13 saved.
Chunk 14 saved.
Chunk 15 saved.
Chunk 16 saved.
Chunk 17 saved.
Chunk 18 saved.
Chunk 19 saved.
Chunk 20 saved.
Chunk 21 saved.
Chunk 22 saved.
Chunk 23 saved.
Chunk 24 saved.
Chunk 25 saved.
Chunk 26 saved.
Chunk 27 saved.
Chunk 28 saved.
Chunk 29 saved.
Chunk 30 saved.
Chunk 31 saved.
Chunk 32 saved.
Chunk 33 saved.
Chunk 34 saved.
Chunk 35 saved.
Chunk 36 saved.
Chunk 37 saved.
Chunk 38 saved.
Chunk 39 saved.
Chunk 40 saved.
Chunk 41 saved.
Chunk 42 saved.
Chunk 43 saved.
Chunk 44 saved.
Chunk 45 saved.
Chunk 46 saved.
Chunk 47 saved.
Chunk 48 saved.
Chunk 49 saved.
Chunk 50 saved.
Chunk 51 saved.
Chunk 52 saved.
Matrix split and saved as JSON chunks.


saving the metadata similarity matrix into 52 different chunks

In [3]:
import numpy as np
import json

similarity_matrix = np.load("adjusted_similarity_matrix.npy")
subset = similarity_matrix[:100, :100]  # First 100 rows and columns

with open("similarity_matrix_subset.json", "w") as f:
    json.dump(subset.tolist(), f)

print("Subset saved.")

Subset saved.


Saving a subset of similarity matrix (First 100 rows and columns) into a json file